In [ ]:
import numpy as np
import pandas as pd

# Install CatBoost (only needs to be run once)
!pip install catboost

# Load datasets
df1 = pd.read_csv('Tuesday-WorkingHours.pcap_ISCX.csv', low_memory=True)
df2 = pd.read_csv('Wednesday-workingHours.pcap_ISCX.csv', low_memory=True)
df3 = pd.read_csv('Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv', low_memory=True)

# Concatenate dataframes
dataset = pd.concat([df1, df2, df3], ignore_index=True)

# Clean column names
dataset.columns = dataset.columns.str.strip()

# Separate features (X) and target (y)
X = dataset.iloc[:, :-1]
y = dataset.iloc[:, -1]

# Convert features to numeric, coercing errors
X = X.apply(pd.to_numeric, errors='coerce')

# Replace infinite values with NaN
X.replace([np.inf, -np.inf], np.nan, inplace=True)

# Impute missing values with the mean
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(missing_values=np.nan, strategy='mean')
X = imputer.fit_transform(X)

# Encode target labels
from sklearn.preprocessing import LabelEncoder
labelencoder_y = LabelEncoder()
y = labelencoder_y.fit_transform(y)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.kernel_approximation import Nystroem
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, classification_report

# Classifiers
from catboost import CatBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

# Define the lists for test_size and kernel types
test_sizes = [0.2, 0.4, 0.6]
kernels = ['linear', 'sigmoid', 'rbf', 'cosine', 'poly']

algorithms = {
    'CatBoost': CatBoostClassifier(verbose=False), # Set verbose to False to suppress training output
    'DecisionTreeClassifier': DecisionTreeClassifier(criterion='entropy', random_state=0),
    'GradientBoostingClassifier': GradientBoostingClassifier(),
    'KNeighborsClassifier': KNeighborsClassifier(n_neighbors=3),
    'LinearDiscriminantAnalysis': LinearDiscriminantAnalysis()
}

In [9]:
results = []

for test_size in test_sizes:
    print(f"\n{'='*60}\nRunning with Test Size: {test_size*100:.0f}%\n{'='*60}")

    # Split data once for the current test_size
    X_train_split, X_test_split, y_train_split, y_test_split = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=0,
        stratify=y
    )

    for kernel in kernels:
        print(f"\n{'*'*50}\n  Applying Nystroem with Kernel: {kernel}\n{'*'*50}")

        # Apply Nystroem kernel approximation
        # Adjust gamma/coef0 for sigmoid/poly if needed, using default for now
        if kernel == 'sigmoid':
          k_pca = Nystroem(n_components=2, kernel=kernel, gamma=15, coef0=1, random_state=0)
        elif kernel == 'poly':
          k_pca = Nystroem(n_components=2, kernel=kernel, degree=3, coef0=1, random_state=0) # Default degree 3 for poly
        else:
          k_pca = Nystroem(n_components=2, kernel=kernel, random_state=0)

        X_train_transformed = k_pca.fit_transform(X_train_split)
        X_test_transformed = k_pca.transform(X_test_split)

        for algo_name, classifier_model in algorithms.items():
            print(f"\n--- Running Algorithm: {algo_name} ---")

            # For KNeighborsClassifier, create a new instance with n_neighbors=3 each time
            # to avoid re-fitting a previously fitted model.
            if algo_name == 'KNeighborsClassifier':
                classifier = KNeighborsClassifier(n_neighbors=3)
            else:
                classifier = classifier_model

            try:
                classifier.fit(X_train_transformed, y_train_split)
                y_pred = classifier.predict(X_test_transformed)

                # Evaluate and print metrics
                cm = confusion_matrix(y_test_split, y_pred)
                accuracy = accuracy_score(y_test_split, y_pred)
                precision = precision_score(y_test_split, y_pred, average='weighted', zero_division=0)
                recall = recall_score(y_test_split, y_pred, average='weighted', zero_division=0)
                f1 = f1_score(y_test_split, y_pred, average='weighted', zero_division=0)
                report = classification_report(y_test_split, y_pred, zero_division=0)

                print(f"Confusion Matrix for {algo_name} (Test Size: {test_size*100:.0f}%, Kernel: {kernel}):\n{cm}")
                print(f"Accuracy: {accuracy:.4f}")
                print(f"Precision: {precision:.4f}")
                print(f"Recall: {recall:.4f}")
                print(f"F1 Score: {f1:.4f}")
                print(f"Classification Report:\n{report}")
                print("---------------------------------------------------")

                results.append({
                    'test_size': test_size,
                    'kernel': kernel,
                    'algorithm': algo_name,
                    'accuracy': accuracy,
                    'precision': precision,
                    'recall': recall,
                    'f1_score': f1,
                    'confusion_matrix': cm.tolist(), # Convert numpy array to list for storage
                    'classification_report': report
                })

            except Exception as e:
                print(f"Error training {algo_name} with Test Size {test_size*100:.0f}% and Kernel {kernel}: {e}")
                results.append({
                    'test_size': test_size,
                    'kernel': kernel,
                    'algorithm': algo_name,
                    'error': str(e)
                })



Running with Test Size: 20%

**************************************************
  Applying Nystroem with Kernel: linear
**************************************************

--- Running Algorithm: CatBoost ---
Confusion Matrix for CatBoost (Test Size: 20%, Kernel: linear):
[[85345   702   142    71   205   110     0     0     0]
 [ 1218 18188    28     0     0     0     0     0     0]
 [  151    98   840    10     0     1     0     0     0]
 [  297    47    11   804     0     0     0     0     0]
 [  172     0     0     0  1416     0     0     0     0]
 [  149     0     0     0     4   403     0     0     0]
 [  285     0     0     0     0    17     0     0     0]
 [    1     0     0     0     0     3     0     0     0]
 [  129     1     0     0     0     0     0     0     0]]
Accuracy: 0.9652
Precision: 0.9611
Recall: 0.9652
F1 Score: 0.9630
Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.99      0.98     86575
           1  

In [ ]:
print(f"\n{'='*60}\nDisplaying all computed results\n{'='*60}")

for result in results:
    algo_name = result.get('algorithm')
    test_size = result.get('test_size')
    kernel = result.get('kernel')

    if 'error' in result:
        print(f"\n--- Algorithm: {algo_name} (Test Size: {test_size*100:.0f}%, Kernel: {kernel}) ---")
        print(f"Error: {result['error']}")
    else:
        print(f"\n--- Algorithm: {algo_name} (Test Size: {test_size*100:.0f}%, Kernel: {kernel}) ---")
        print(f"Accuracy: {result['accuracy']:.4f}")
        print(f"Precision: {result['precision']:.4f}")
        print(f"Recall: {result['recall']:.4f}")
        print(f"F1 Score: {result['f1_score']:.4f}")
        print(f"Classification Report:\n{result['classification_report']}")
        print("---------------------------------------------------")

In [ ]:
print(f"\n{'='*60}\nSpecific Results for Test Size: 20%, Kernel: sigmoid\n{'='*60}")

for result in results:
    if result.get('test_size') == 0.2 and result.get('kernel') == 'sigmoid':
        algo_name = result.get('algorithm')
        if 'error' in result:
            print(f"\n--- Algorithm: {algo_name} ---")
            print(f"Error: {result['error']}")
        else:
            print(f"\n--- Algorithm: {algo_name} (Test Size: {result['test_size']*100:.0f}%, Kernel: {result['kernel']}) ---")
            print(f"Accuracy: {result['accuracy']:.4f}")
            print(f"Precision: {result['precision']:.4f}")
            print(f"Recall: {result['recall']:.4f}")
            print(f"F1 Score: {result['f1_score']:.4f}")
            print(f"Classification Report:\n{result['classification_report']}")
            print("---------------------------------------------------")

### Summary of Results

All results have been processed and stored in the `results` list. You can inspect this list to programmatically access the performance metrics for each combination of `test_size`, `Nystroem` kernel, and classifier.

For example, to display all collected results, you could convert the `results` list to a DataFrame:


In [ ]:
import pandas as pd
results_df = pd.DataFrame(results)
display(results_df.head()) # Display the first few rows of the results DataFrame


## Pushing to GitHub

To push your Colab notebook to GitHub, you'll generally follow these steps:

1.  **Mount Google Drive**: Your Colab notebooks are stored in your Google Drive. You'll need to mount it to access the `.ipynb` file.
2.  **Clone your GitHub Repository**: You'll need to clone the repository where you want to save your notebook. If you don't have one, create a new empty repository on GitHub first.
3.  **Copy the Notebook to the Repository**: Move the `.ipynb` file from its current location in Drive to your cloned GitHub repository's directory.
4.  **Add, Commit, and Push**: Use Git commands to stage your changes, commit them, and push them to your GitHub repository.

### Step 1: Mount Google Drive

This allows Colab to access files stored in your Google Drive, including this notebook.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Step 2: Navigate to your project directory and clone your GitHub Repository

First, change the directory to where you want to store your GitHub repository, typically within your Google Drive. Then, clone your repository. **Replace `YOUR_GITHUB_USERNAME` and `YOUR_REPOSITORY_NAME` with your actual GitHub username and repository name.**

**Important**: You might need to generate a [Personal Access Token (PAT)](https://docs.github.com/en/authentication/keeping-your-account-and-data-secure/creating-a-personal-access-token) from GitHub and use it instead of your password for cloning. If you want to use SSH, you'll need to configure SSH keys in Colab and GitHub, which is more involved. For simplicity, HTTPS with a PAT is often used.

In [ ]:
# Change to a directory in your Google Drive (e.g., a folder named 'GitHub_Projects')
# Make sure this directory exists in your Google Drive
%cd /content/drive/MyDrive/GitHub_Projects

# Clone your repository
# Replace 'YOUR_GITHUB_USERNAME' and 'YOUR_REPOSITORY_NAME'
# You will be prompted for your GitHub username and Personal Access Token (PAT) or password.
!git clone https://github.com/YOUR_GITHUB_USERNAME/YOUR_REPOSITORY_NAME.git

### Step 3: Copy the notebook into the cloned repository

Colab notebooks are usually saved in `My Drive/Colab Notebooks/`. You'll need to find the exact path of your current notebook. You can find this by looking at the file explorer in Colab or checking your Google Drive.

**Replace `path/to/your/current/notebook.ipynb` with the actual path to this notebook in your Google Drive.**

**Replace `YOUR_REPOSITORY_NAME` with the name of the folder created by cloning your repository.**

In [ ]:
# Navigate into your cloned repository directory
%cd /content/drive/MyDrive/GitHub_Projects/YOUR_REPOSITORY_NAME

# Get the current notebook's path (this is a common path, but verify for your notebook)
notebook_path = '/content/drive/MyDrive/Colab Notebooks/Untitled0.ipynb' # <<-- ADJUST THIS PATH

# You might need to rename 'Untitled0.ipynb' to the actual name of your notebook file
# Copy the notebook into the current directory (your cloned repo)
!cp "{notebook_path}" .

### Step 4: Add, Commit, and Push to GitHub

Now, use standard Git commands to push your notebook to your remote repository.

In [ ]:
# Configure Git (if not already done)
!git config --global user.email "you@example.com" # Replace with your email
!git config --global user.name "Your Name"       # Replace with your name

# Add the notebook file to staging
# If you copied 'Untitled0.ipynb', add that. Otherwise, add the correct filename.
!git add Untitled0.ipynb # <<-- ADJUST THIS FILENAME IF NECESSARY

# Commit the changes
!git commit -m "Add initial Colab notebook" # Use a meaningful commit message

# Push the changes to the remote repository
!git push origin main # Or 'master', depending on your branch name

After running these cells, your notebook should be visible in your GitHub repository! Remember to adjust the placeholders like `YOUR_GITHUB_USERNAME`, `YOUR_REPOSITORY_NAME`, and the notebook file path.